# Query Optimization Demo

In [1]:
# Run this cell to set up imports
import numpy as np
import pandas as pd

## Load in the IMDB Performance database

This is a variation of the IMDB database with keys defined. Note that this is a pretty big database! So if you run the below lines, please also remember to delete the `imdb_perf_lecture` afterwards to save space on your limited postgreSQL server.

We assume you have the associated lecture folder `indexes` pulled into your repo already. The below commands create a symbolic link (i.e., shortcut/redirect with `ln`) to this lecture data directory, allowing some space saving, and unzip the database file.

In [2]:
!ln -sf ../../lec/indexes/data .
!unzip -u data/imdb_perf_lecture.zip -d data/

Archive:  data/imdb_perf_lecture.zip


In [3]:
!psql -h localhost -c 'DROP DATABASE IF EXISTS imdb_perf_lecture'
!psql -h localhost -c 'CREATE DATABASE imdb_perf_lecture' 
!psql -h localhost -d imdb_perf_lecture -f data/imdb_perf_lecture.sql

DROP DATABASE
CREATE DATABASE
SET
SET
SET
SET
SET
 set_config 
------------
 
(1 row)

SET
SET
SET
SET
SET
SET
CREATE TABLE
ALTER TABLE
CREATE TABLE
ALTER TABLE
CREATE TABLE
ALTER TABLE
COPY 845888
COPY 2211936
COPY 656453
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE


## Start `jupysql`

In [4]:
%reload_ext sql

In [5]:
%sql postgresql://127.0.0.1:5432/imdb_perf_lecture

Connecting to 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

If you're having trouble seeing the entirety of query plans, you can run the following cell to set the limit on displayed rows to 20. **Careful**: Do not set this to `None` and run the actual queries; SQL will return millions of rows and crash your kernel!

In [6]:
# run this cell to remove 10-row limit on display
%config SqlMagic.displaylimit = 20

## Single Table Plans: Impact of limits, orders, selections




<div class="alert alert-success">
It is much easier to see query plans in <b>psql</b>!<br/>
<code>jupysql</code> dataframe visualization removes any whitespace.
</div>

We add a printing helper function that tries to do the same thing and preserves indentation.
Use `explain("SELECT ...")` to do `EXPLAIN ANALYZE` of the query.

In [13]:
import html
from IPython import get_ipython
from IPython.display import display, HTML

def printplans(x):
    text = "\n".join(line for (line,) in x)
    display(HTML(f'<pre style="white-space: pre; overflow-x: auto">{html.escape(text)}</pre>'))

def explain(query, analyze=True):
    prefix = "EXPLAIN ANALYZE " if analyze else "EXPLAIN "
    printplans(get_ipython().run_line_magic("sql", prefix + query))

<br/>

Let's start with a simple query. Equivalent `explain(..)` shown next. 

In [9]:
%%sql
/* 1a */
EXPLAIN ANALYZE SELECT id FROM actors;

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

3 rows affected.

QUERY PLAN
Seq Scan on actors (cost=0.00..13684.88 rows=845888 width=4) (actual time=0.064..63.252 rows=845888 loops=1)
Planning Time: 0.076 ms
Execution Time: 91.275 ms


In [16]:
# 1a
explain ("SELECT id FROM actors;")

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

3 rows affected.

<br/>

Great, now let's see the impact of adding a LIMIT clause. 

In [17]:
# 1b 
explain("SELECT id FROM actors LIMIT 10;")

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

4 rows affected.

<br/>

Wow - that dropped quite a bit!

What if we add an ORDER BY?

In [21]:
# 1c
explain("SELECT id FROM actors ORDER BY name LIMIT 10;")

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

12 rows affected.

<br/> 

The time goes up again! Sorting is expensive - even if all we're doing is getting the top k.



Finally, what if we added a selection:

In [23]:
# 1d
explain("SELECT id FROM actors WHERE id > 4000000 AND name='Tom Hanks';")

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

8 rows affected.

Here, the selection is as part of sequential scan (see "Filter:" )

<br/> <br/>

## Two-table demo: LIMIT

Let's join two tables, `actors` and `cast_info`. The query planner selects a hash join:

In [27]:
explain('''
SELECT * FROM actors, cast_info 
WHERE actors.id = cast_info.person_id;
''')

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

8 rows affected.

<br/>

Below, we add `LIMIT`. Note the query planner switches to a nested loop join, using an index scan to match `cast_info.person_id` to the indexed attribute `actors.id`! This results in a 1000x speedup!

Thus, knowing which keywords impact performance is really, really important!

In [29]:
explain('''
SELECT * 
FROM actors, cast_info 
WHERE actors.id = cast_info.person_id 
LIMIT 10;''')

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

11 rows affected.

<br/> 

## Two-table demo: Selection

Let's return to the old query and see the impact of a selection.

In [31]:
explain('''
SELECT * FROM actors, cast_info 
WHERE actors.id = cast_info.person_id;
''')

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

8 rows affected.

Now let's add an additional predicate. 

In [49]:
explain('''
SELECT * FROM actors, cast_info 
WHERE actors.id = cast_info.person_id AND actors.id > 300000000;
''')

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

12 rows affected.

Time comes down drastically, plus predicate is evaluated while actor is scanned!
At home, try changing 30000000 to 3000, 30000, 300000, 3000000.


At home, compare the following query, where we restrict the attributes in `SELECT`, to the previous. What has changed?

In [32]:
explain('''
SELECT actors.name FROM actors, cast_info 
WHERE actors.id = cast_info.person_id AND actors.id > 30000000;
''')

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

12 rows affected.

## [At Home, or in class if time] Three-way joins

Now let's try a three-way join! Feel free to change the LIMIT, but note that removing it entirely can result in a really long query! <br/> <br/>
We've set it to be a large number, 100K, to start.

In [54]:
explain('''
SELECT *
FROM actors, cast_info, movies
WHERE actors.id = cast_info.person_id
    AND movies.id = cast_info.movie_id
LIMIT 100000;
''')

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

17 rows affected.

<br/><br/>
At home, try the impact of changing 100K to a smaller numner.

What if we add an additional selection condition on one of the relations?

Below, note the predicate pushdown in the sequential scan on actors! Again, copy-paste into `psql` if you can't see the whitespace formatting.

In [55]:
explain('''
SELECT *
FROM actors, cast_info, movies
WHERE actors.id = cast_info.person_id
    AND movies.id = cast_info.movie_id
    AND name = 'Tom Hanks';
''')

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

16 rows affected.

<br/><br/>

Compare with the below predicate pushdown, where the filter is now on movie titles:

In [52]:
explain('''
SELECT *
FROM actors, cast_info, movies
WHERE actors.id = cast_info.person_id
    AND movies.id = cast_info.movie_id
    AND title LIKE 'Snakes on a Plane';
    ''')

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

16 rows affected.

# [At home] Three-way joins with Indexes 

We're now going to see the impact of indexes... Start with a query from before.

In [56]:
explain('''
SELECT *
FROM actors, cast_info, movies
WHERE actors.id = cast_info.person_id
    AND movies.id = cast_info.movie_id
LIMIT 10;
''')

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

14 rows affected.

<br/><br/>
What if we dropped one of the indexes?

To do so we must drop the primary key constraint on actors.id:

In [57]:
%sql ALTER TABLE actors DROP CONSTRAINT actor_pkey CASCADE;

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

++
||
++
++

In [58]:
explain('''
SELECT *
FROM actors, cast_info, movies
WHERE actors.id = cast_info.person_id
    AND movies.id = cast_info.movie_id
LIMIT 10;
''')

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

15 rows affected.

<br/><br/>
What if we dropped both indexes?

In [59]:
%sql ALTER TABLE movies DROP CONSTRAINT movie_pkey CASCADE;

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

++
||
++
++

In [60]:
explain('''
SELECT *
FROM actors, cast_info, movies
WHERE actors.id = cast_info.person_id
    AND movies.id = cast_info.movie_id
LIMIT 10;
''')

Running query in 'postgresql://127.0.0.1:5432/imdb_perf_lecture'

17 rows affected.

# Cleanup

We close the connection, then drop the database:

In [61]:
%sql --close postgresql://127.0.0.1:5432/imdb_perf_lecture

In [62]:
!psql -h localhost -c 'DROP DATABASE IF EXISTS imdb_perf_lecture'

DROP DATABASE
